# 急减速候选挖掘

读取同目录的虚构车辆事件，筛选指定日期内达到速度下降阈值的候选，并输出 `scene-manifest.csv`。

In [ ]:
event_date = "2026-08-12"
threshold_kmh = 20
window_seconds = 2

In [ ]:
from pathlib import Path
import csv

input_path = Path("vehicle-events.csv")
with input_path.open(newline="", encoding="utf-8") as source:
    events = list(csv.DictReader(source))

print(f"loaded={len(events)} from {input_path.resolve()}")

In [ ]:
def speed_drop_kmh(event):
    return int(event["speed_before_kmh"]) - int(event["speed_after_kmh"])

candidates = [
    event
    for event in events
    if event["event_date"] == event_date
    and int(event["window_seconds"]) == window_seconds
    and speed_drop_kmh(event) >= threshold_kmh
]

candidate_ids = [event["event_id"] for event in candidates]
print(f"candidates={candidate_ids}")

In [ ]:
output_path = Path("scene-manifest.csv")
fieldnames = [
    "event_id",
    "event_date",
    "vehicle_id",
    "speed_drop_kmh",
    "raw_log_path",
]

with output_path.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()
    for event in candidates:
        writer.writerow({
            "event_id": event["event_id"],
            "event_date": event["event_date"],
            "vehicle_id": event["vehicle_id"],
            "speed_drop_kmh": speed_drop_kmh(event),
            "raw_log_path": event["raw_log_path"],
        })

print(f"wrote={len(candidates)} to {output_path.resolve()}")

In [ ]:
print(output_path.read_text(encoding="utf-8"))